# 06 — Understand the classifier's mistakes

This notebook reads saved training-data out-of-fold predictions. It does not fit models or read the reserved test set. A **false negative** is a fraudulent ad the classifier missed; a **false positive** is a genuine ad it flagged.

We compare A2 logistic 30k, word + character logistic, and LightGBM 2k. Every posting is scored once in each of five repeats. Those are repeated views of the same ad, not five independent examples. Predictions use each fold's saved training-chosen cutoff, not a newly chosen threshold.

Workflow: run the cells, inspect a diverse sample of errors, record observations, and choose one focused next experiment. Development error analysis can influence later model choices, so it is exploratory evidence rather than fresh validation.

In [1]:
from pathlib import Path
import html
import numpy as np
import pandas as pd
from IPython.display import display, HTML

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/evaluation.py').exists())
DATA = ROOT / 'data/processed'
train = pd.read_csv(DATA / 'd1_train.csv')
assignments = pd.read_csv(DATA / 'model_cv_assignments.csv')
assert train.job_id.is_unique
frames = []
# Reconstruct the saved baseline cutoffs; the original 02 OOF file contains probabilities only.
pred = pd.read_csv(DATA / 'model_logreg_a2_oof.csv')
pred = pred.merge(assignments[['repeat', 'fold', 'job_id']], on=['repeat', 'job_id'], validate='one_to_one')
scores = pd.read_csv(ROOT / 'reports/tables/model_logreg_a2_fold_scores.csv')
pred = pred.merge(scores[['repeat', 'fold', 'threshold']], on=['repeat', 'fold'], validate='many_to_one')
frames.append(pred.assign(model='A2 logistic 30k'))
for name, path in {
    'Word + character logistic': DATA / 'classifier_word_char/logreg_word_char_predictions.csv',
    'LightGBM A2 2k': DATA / 'model_comparison_revised/lightgbm_a2_2k_predictions.csv',
}.items():
    frames.append(pd.read_csv(path).assign(model=name))
predictions = pd.concat(frames, ignore_index=True)
if predictions.duplicated(['model', 'repeat', 'job_id']).any():
    raise ValueError('Duplicate saved predictions.')
expected_ids = set(train.job_id)
for (model, repeat), batch in predictions.groupby(['model', 'repeat']):
    if set(batch.job_id) != expected_ids:
        raise ValueError(f'Incomplete predictions for {model}, repeat {repeat}.')
if not np.isfinite(predictions[['probability', 'threshold']]).all().all():
    raise ValueError('Invalid saved scores.')
checks = predictions.merge(assignments[['repeat', 'fold', 'job_id']], on=['repeat', 'job_id'], suffixes=('', '_expected'), validate='many_to_one')
assert checks.fold.eq(checks.fold_expected).all(), 'Saved outer assignments differ.'
predictions = predictions.merge(train[['job_id', 'group_id', 'fraudulent']], on='job_id', validate='many_to_one')
predictions['flagged'] = predictions.probability >= predictions.threshold
predictions['FN'] = predictions.fraudulent.eq(1) & ~predictions.flagged
predictions['FP'] = predictions.fraudulent.eq(0) & predictions.flagged
print(f'{len(train):,} training ads; {predictions.model.nunique()} models. No training or test-set access.')

14,439 training ads; 3 models. No training or test-set access.


## 1. How often does each kind of error happen?

The table averages posting counts over repeats. It does not add the five repeats into a larger dataset. These pooled-per-repeat rates can differ from the mean-of-fold rates reported in the modelling notebooks because aggregation weights differ.

In [2]:
rows = []
for (model, repeat), batch in predictions.groupby(['model', 'repeat']):
    tp = int((batch.flagged & batch.fraudulent.eq(1)).sum())
    fp, fn = int(batch.FP.sum()), int(batch.FN.sum())
    rows.append({'model': model, 'repeat': repeat, 'fraud_caught': tp, 'fraud_missed': fn,
                 'genuine_flagged': fp, 'precision': tp / max(tp + fp, 1), 'recall': tp / max(tp + fn, 1)})
display(pd.DataFrame(rows).groupby('model').mean(numeric_only=True).drop(columns='repeat').round(3))

,fraud_caught,fraud_missed,genuine_flagged,precision,recall
model,,,,,
A2 logistic 30k,370.8,322.2,250.2,0.597,0.535
LightGBM A2 2k,376.0,317.0,378.2,0.513,0.543
Word + character logistic,357.8,335.2,273.4,0.568,0.516


## 2. Which ads are repeatedly misclassified?

Select a model below. `error_repeats` counts how many repeats got that ad wrong. Start with persistent errors (at least 4 of 5 repeats). Keep one posting per group in the review sample so copied templates do not dominate. Include errors unique to a model and errors shared by all models.

In [3]:
MODEL = 'Word + character logistic'
ERROR_TYPE = 'FN'  # 'FN' = missed fraud; 'FP' = genuine ad wrongly flagged
MIN_ERROR_REPEATS = 4
SAMPLE_SIZE = 10

per_ad = predictions.groupby(['model', 'job_id']).agg(
    error_FN=('FN', 'sum'), error_FP=('FP', 'sum'), repeats=('repeat', 'nunique'),
    mean_score=('probability', 'mean'), mean_cutoff=('threshold', 'mean')).reset_index()
per_ad['error_repeats'] = per_ad[f'error_{ERROR_TYPE}']
wide = per_ad.pivot(index='job_id', columns='model', values='error_repeats')
shared = wide.ge(MIN_ERROR_REPEATS).sum(axis=1).rename('models_with_persistent_error')
cases = per_ad[per_ad.model.eq(MODEL) & per_ad.error_repeats.ge(MIN_ERROR_REPEATS)].copy()
cases = cases.merge(train, on='job_id', validate='one_to_one').merge(shared, on='job_id')
# Randomise within error persistence, then retain one ad per group.
sample = (cases.sample(frac=1, random_state=42)
          .sort_values('error_repeats', ascending=False, kind='stable')
          .drop_duplicates('group_id').head(SAMPLE_SIZE))
print(f'{len(cases)} persistent {ERROR_TYPE} ads in {cases.group_id.nunique()} groups; showing {len(sample)} distinct groups.')
display(sample[['job_id', 'group_id', 'title', 'fraudulent', 'error_repeats', 'repeats', 'models_with_persistent_error', 'mean_score', 'mean_cutoff']])
print('Mean scores/cutoffs are descriptive. Error counts use the original per-fold cutoffs, not their averages.')

316 persistent FN ads in 125 groups; showing 10 distinct groups.


,job_id,group_id,title,fraudulent,error_repeats,repeats,models_with_persistent_error,mean_score,mean_cutoff
173,8678,309,Project Subcontracting Manager,1,5,5,3,0.307067,0.589734
165,8058,2769,Marketing/sales Coordinator,1,5,5,2,0.145540,0.536075
78,4590,615,Web design intern,1,5,5,3,0.131408,0.580524
93,5363,2032,Licensed Life Insurance Agent,1,5,5,3,0.250741,0.579091
262,17663,4342,iSeries Team Lead,1,5,5,3,0.293672,0.569261
25,1961,294,Mobile Developer-Back End,1,5,5,3,0.150404,0.551982
63,3882,1605,Earn a nice part time PAY!,1,5,5,2,0.509436,0.596342
82,4606,418,Collection Representative II,1,5,5,3,0.336319,0.601533
60,3619,1520,"Agency Sales Managers $150-$175,000/yr",1,5,5,3,0.095093,0.552178
146,6855,2131,Network Marketing,1,5,5,3,0.303578,0.582382


Mean scores/cutoffs are descriptive. Error counts use the original per-fold cutoffs, not their averages.


## 3. Read individual postings and compare all model outcomes

Change `JOB_ID` to one of the IDs above. Read the full fields, not just the title. Look for: ambiguous wording, a suspicious request, ordinary wording shared with scams, sparse employer information, unusual formatting, truncation risks for future embedding models, or a potentially questionable dataset label. A dataset label is not an independent verification of the employer. Scores are model outputs, not verified fraud probabilities.

In [7]:
# JOB_ID = int(sample.iloc[0].job_id) if len(sample) else None
JOB_ID = 4090
if JOB_ID is None:
    print('No matching sample. Lower MIN_ERROR_REPEATS or change MODEL / ERROR_TYPE.')
else:
    posting = train.set_index('job_id').loc[JOB_ID]
    display(per_ad[per_ad.job_id.eq(JOB_ID)][['model', 'error_FN', 'error_FP', 'repeats', 'mean_score', 'mean_cutoff']])
    parts = [f'<h3>Job {JOB_ID}: {html.escape(str(posting.title))}</h3>']
    for field in ['fraudulent', 'group_id', 'location', 'salary_range', 'has_company_logo', 'title', 'company_profile', 'description', 'requirements', 'benefits']:
        value = posting.get(field)
        value = '(missing)' if pd.isna(value) else str(value)
        parts.append(f'<h4>{html.escape(field)}</h4><div style="white-space:pre-wrap">{html.escape(value)}</div>')
    display(HTML(''.join(parts)))

,model,error_FN,error_FP,repeats,mean_score,mean_cutoff
3236,A2 logistic 30k,0,0,5,0.019239,0.749364
17675,LightGBM A2 2k,0,0,5,0.000207,0.330097
32114,Word + character logistic,0,0,5,0.017632,0.546689


## 4. Record observations, then choose one experiment

Create a review sheet for the selected sample. Edit the CSV locally or record notes here. Re-running this cell preserves an existing sheet. Change `ERROR_TYPE` to review both missed fraud and false alarms; inspect some correctly classified ads too, so a pattern common to every ad is not mistaken for an explanation.

Suggested categories: `semantic_or_paraphrase`, `ordinary_job_language`, `missing_information`, `formatting_or_spelling`, `label_question`, `unclear`. Write an exact supporting quote and why it matters. These are hypotheses about errors, not proven causal explanations of the score.

Decision examples: paraphrased scams missed by all models → test semantic embeddings; missing structural information → investigate input features; many uncertain labels → review annotation quality. Do not relabel examples merely because the classifier disagrees. Do not inspect the reserved test set to choose the next model.

In [5]:
review_dir = DATA / 'error_analysis'
review_dir.mkdir(parents=True, exist_ok=True)
review_path = review_dir / f"{MODEL.lower().replace(' ', '_').replace('+', 'plus')}_{ERROR_TYPE.lower()}_review.csv"
if not review_path.exists():
    review = sample[['job_id', 'group_id', 'title', 'fraudulent', 'error_repeats', 'models_with_persistent_error']].copy()
    for column in ['category', 'supporting_field', 'supporting_quote', 'observation', 'candidate_experiment']:
        review[column] = ''
    review.to_csv(review_path, index=False)
    print('Created review sheet:', review_path)
else:
    print('Preserved existing review sheet:', review_path)

Created review sheet: /Users/yiting/Desktop/nus/y4s1/bt4012/group project/data/processed/error_analysis/word_plus_character_logistic_fn_review.csv
